In [4]:
import pandas as pd
import numpy as np

## Full_df_filtered 

In [5]:
full_df = pd.read_csv('full_df_filtered.csv', low_memory=False)

# total images per class
images_per_class = (
    full_df.groupby("species")
      .size()
      .reset_index(name="n_images")
      .sort_values("n_images", ascending=False)
)

images_per_class.to_csv(
    "images_per_class_main.csv",
    index=False
)

In [6]:
images_per_species_site = (
    full_df.groupby(["site", "species"])
      .size()
      .reset_index(name="n_images")
      .sort_values(["site", "n_images"], ascending=[True, False])
)

images_per_species_site.to_csv(
    "images_per_species_per_site_main.csv",
    index=False
)

## S6 sheet

In [7]:
df_s6 = pd.read_csv("../WLD_S6_report_species(S6 Species Cleaning).csv")

# define invalid correction values
invalid_corrections = ["?", "\\", "", np.nan]

# create mask: valid correct_species
mask = (
    df_s6["correct_species"].notna() &
    ~df_s6["correct_species"].isin(invalid_corrections)
)

# overwrite question__species where mask is True
df_s6.loc[mask, "question__species"] = df_s6.loc[mask, "correct_species"]

/tmp/ipykernel_92466/2215681646.py:1: DtypeWarning: Columns (13,14,17,19,26,40,42) have mixed types. Specify dtype option on import or set low_memory=False.
  df_s6 = pd.read_csv("../WLD_S6_report_species(S6 Species Cleaning).csv")


In [8]:
# 1) rename / standardize species labels
rename_map = {
    "vervetmonkey": "vervet_monkey",
    "groundhornbill": "ground_hornbill",
    "honeybadger": "honey_badger",
    "birdother": "bird_other",
    "reptilesamphibians": "reptile",
    "reptileamphibian": "reptile",
    "reptilesandamphibians": "reptile",
    "sableantelope": "sable_antelope",
    "samangomonkey": "samango_monkey",
    "commonduiker": "duiker_common",
    "redduiker": "duiker_red", 
    "whitetailedmongoose": "mongoose_white_tailed",
    "slendermongoose":"mongoose_slender",
    "bandedmongoose": "mongoose_banded",
    "marshmongoose": "mongoose_marsh",
    "largegreymongoose": "mongoose_large_grey", 
    "bushytailedmongoose": "mongoose_bushy_tailed",
    "thicktailed bushbaby": "bushbaby", 
    "egyptianmongoose": "mongoose_egyptian",
    "civeet": "civet", 
    "lionfemale":"lion",
    "lionmale": "lion", 
    "Impala":"impala"
}

df_s6["question__species"] = (
    df_s6["question__species"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace(rename_map)
)

# 2) remove unwanted / invalid classes
remove_classes = ["0", "1", "flood", "nan", "human", "rodent", "unknown", "setup", "person", "nothing", "maintenance", "Vehicles/Humans/Livestock", "vehicles/humans/livestock", "bat", "unclassifiable"]

df_s6 = df_s6[
    ~df_s6["question__species"].isin(remove_classes)
]

In [9]:
images_per_species_s6 = (
    df_s6.groupby("question__species")
         .size()
         .reset_index(name="n_images")
         .sort_values("n_images", ascending=False)
)

images_per_species_s6.to_csv(
    "images_per_class_s6.csv",
    index=False
)

In [10]:
images_per_species_site_s6 = (
    df_s6.groupby(["site", "question__species"])
         .size()
         .reset_index(name="n_images")
         .sort_values(["site", "n_images"], ascending=[True, False])
)
images_per_species_site_s6.to_csv(
    "images_per_class_per_site_s6.csv",
    index=False
)

## S5 Sheet

In [11]:
df_s5 = pd.read_csv("../S5_speciesID_check(S5_speciesID_check).csv")

# define invalid correction values
invalid_corrections = ["?", "\\", "", np.nan]

# create mask: valid correct_species
mask = (
    df_s5["correct_species"].notna() &
    ~df_s5["correct_species"].isin(invalid_corrections)
)

# overwrite question__species where mask is True
df_s5.loc[mask, "species"] = df_s5.loc[mask, "correct_species"]

/tmp/ipykernel_92466/136233445.py:1: DtypeWarning: Columns (14,16,19) have mixed types. Specify dtype option on import or set low_memory=False.
  df_s5 = pd.read_csv("../S5_speciesID_check(S5_speciesID_check).csv")


In [102]:
df_s5["species"] = (
    df_s5["species"]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace(rename_map)
)
df_s5 = df_s5[
    ~df_s5["species"].isin(remove_classes)
]

In [104]:
images_per_species_s5 = (
    df_s5.groupby("species")
         .size()
         .reset_index(name="n_images")
         .sort_values("n_images", ascending=False)
)
images_per_species_s5.to_csv(
    "images_per_class_s5.csv",
    index=False
)

In [106]:
images_per_species_site_s5 = (
    df_s5.groupby(["site", "species"])
         .size()
         .reset_index(name="n_images")
         .sort_values(["site", "n_images"], ascending=[True, False])
)
images_per_species_site_s5.to_csv(
    "images_per_class_per_site_s5.csv",
    index=False
)

## Download Species

In [15]:
import os
import re
import pandas as pd
import requests
from tqdm import tqdm
from urllib.parse import urlparse

# ---------------------------
# CONFIG
# ---------------------------
BASE_DIR = "/mnt/sharedstorage/sabdelazim/"  # change to a writable path if needed
S5_OUT = os.path.join(BASE_DIR, "S5")
S6_OUT = os.path.join(BASE_DIR, "S6")
os.makedirs(S5_OUT, exist_ok=True)
os.makedirs(S6_OUT, exist_ok=True)

S5_CSV = "../S5_speciesID_check(S5_speciesID_check).csv"
S6_CSV = "../WLD_S6_report_species(S6 Species Cleaning).csv"

URL_COL = "zooniverse_url_0"

# Classes shown in your screenshot
KEEP_CLASSES = {
    "duiker_red",
    "hartebeest",
    "mongoose_marsh",
    "duiker_common",
    "mongoose_white_tailed",
    "mongoose_slender",
    "mongoose_banded",
    "mongoose_bushy_tailed",
    "mongoose_large_grey",
    "eland",
    "duiker",
    "hippopotamus",
    "mongoose_dwarf",
    "pangolin",
}

# Which column is the class label in each file
S5_CLASS_COL = "species"
S6_CLASS_COL = "question__species"  # if your S6 file uses "species", change this to "species"

# ---------------------------
# HELPERS
# ---------------------------
def slugify(x, maxlen=80):
    x = "" if pd.isna(x) else str(x)
    x = x.strip().lower()
    x = re.sub(r"\s+", "_", x)
    x = re.sub(r"[^a-z0-9_\-\.]", "", x)
    return x[:maxlen] if len(x) > maxlen else x

def token_after_last_hyphen(url: str) -> str:
    """
    Extract the last path segment (no query), remove extension,
    then take substring after the last '-' (hyphen).
    """
    path = urlparse(url).path
    base = os.path.basename(path)            # e.g. abc-def-12345.jpeg
    base_no_ext = os.path.splitext(base)[0]  # abc-def-12345
    token = base_no_ext.split("-")[-1]       # 12345
    token = token if token else base_no_ext
    token = slugify(token, maxlen=120)
    return token or "image"

def clear_downloaded_images(folder: str):
    """
    Delete previously downloaded image files in folder.
    Keeps CSV logs (failed_urls.csv, etc).
    """
    exts = {".jpg", ".jpeg", ".png", ".webp"}
    removed = 0
    for fn in os.listdir(folder):
        fp = os.path.join(folder, fn)
        if os.path.isfile(fp) and os.path.splitext(fn.lower())[1] in exts:
            os.remove(fp)
            removed += 1
    print(f"🧹 Removed {removed} images from {folder}")

def download_image(url: str, out_path: str):
    try:
        headers = {"User-Agent": "Mozilla/5.0"}
        with requests.get(url, stream=True, timeout=60, headers=headers) as r:
            if r.status_code != 200:
                return False, f"status={r.status_code}"
            ctype = (r.headers.get("Content-Type") or "").lower()
            if "text/html" in ctype:
                return False, f"content-type={ctype}"
            with open(out_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
        return True, "ok"
    except Exception as e:
        return False, str(e)

def filter_keep_classes(df: pd.DataFrame, class_col: str) -> pd.DataFrame:
    df = df.copy()
    df[class_col] = df[class_col].astype(str).str.strip().str.lower()
    return df[df[class_col].isin(KEEP_CLASSES)]

def download_df(df: pd.DataFrame, out_dir: str, tag: str, class_col: str):
    failed = []
    downloaded = 0
    skipped_blank = 0

    if URL_COL not in df.columns:
        raise ValueError(f"Missing '{URL_COL}' in {tag}. Columns: {list(df.columns)}")
    if "site" not in df.columns:
        raise ValueError(f"Missing 'site' in {tag}. Columns: {list(df.columns)}")
    if class_col not in df.columns:
        raise ValueError(f"Missing '{class_col}' in {tag}. Columns: {list(df.columns)}")

    for i, row in tqdm(df.iterrows(), total=len(df), desc=f"Downloading {tag}"):
        url = row.get(URL_COL)
        if pd.isna(url) or str(url).strip() == "":
            skipped_blank += 1
            continue

        cls = slugify(row.get(class_col, "class"))
        site = slugify(row.get("site", "site"))

        tok = token_after_last_hyphen(str(url).strip())
        fname = f"{tok}_{site}_{cls}.jpeg"
        out_path = os.path.join(out_dir, fname)

        # if you re-run, don't redownload the same filename
        if os.path.exists(out_path):
            continue

        ok, reason = download_image(str(url).strip(), out_path)
        if ok:
            downloaded += 1
        else:
            failed.append({"row": i, "url": str(url).strip(), "reason": reason})
            if os.path.exists(out_path):
                try:
                    os.remove(out_path)
                except Exception:
                    pass

    if failed:
        pd.DataFrame(failed).to_csv(os.path.join(out_dir, "failed_urls.csv"), index=False)

    print(f"\n✅ {tag} finished → {out_dir}")
    print(f"Downloaded: {downloaded}")
    print(f"Blank URLs skipped: {skipped_blank}")
    print(f"Failed: {len(failed)} (see failed_urls.csv if >0)")

# ---------------------------
# RUN
# ---------------------------
# 1) remove already-downloaded images
clear_downloaded_images(S5_OUT)
clear_downloaded_images(S6_OUT)

# 2) read CSVs
df_s5 = pd.read_csv(S5_CSV, low_memory=False)
df_s6 = pd.read_csv(S6_CSV, low_memory=False)

# 3) filter to only screenshot classes
df_s5_f = filter_keep_classes(df_s5, S5_CLASS_COL)
df_s6_f = filter_keep_classes(df_s6, S6_CLASS_COL)

print("S5 rows after filtering:", len(df_s5_f))
print("S6 rows after filtering:", len(df_s6_f))

# 4) download only those classes
download_df(df_s5_f, S5_OUT, "S5", S5_CLASS_COL)
download_df(df_s6_f, S6_OUT, "S6", S6_CLASS_COL)

print("\nImages saved in:")
print(S5_OUT)
print(S6_OUT)

🧹 Removed 1360 images from /mnt/sharedstorage/sabdelazim/S5
🧹 Removed 0 images from /mnt/sharedstorage/sabdelazim/S6
S5 rows after filtering: 626
S6 rows after filtering: 488



✅ S5 finished → /mnt/sharedstorage/sabdelazim/S5
Downloaded: 626
Blank URLs skipped: 0
Failed: 0 (see failed_urls.csv if >0)



✅ S6 finished → /mnt/sharedstorage/sabdelazim/S6
Downloaded: 488
Blank URLs skipped: 0
Failed: 0 (see failed_urls.csv if >0)

Images saved in:
/mnt/sharedstorage/sabdelazim/S5
/mnt/sharedstorage/sabdelazim/S6


In [19]:
import os
import pandas as pd

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

def parse_filename(filename):
    """
    Expected format:
    something_something_SITE_species_name.jpeg
    or
    token_SITE_species_name.jpeg

    Returns: (species, site) or (None, None)
    """
    base = os.path.splitext(filename)[0]
    parts = base.split("_")

    site_pat = re.compile(r"^[a-zA-Z]\d{2}$")

    for i, part in enumerate(parts):
        if site_pat.match(part):
            site = part.lower()
            species = "_".join(parts[i + 1:]).lower()
            return species, site

    return None, None


def parse_folder(folder):
    rows = []

    for fn in os.listdir(folder):
        if os.path.splitext(fn.lower())[1] not in IMAGE_EXTS:
            continue

        species, site = parse_filename(fn)

        if species is None:
            continue

        rows.append({
            "site": site,
            "species": species
        })

    return pd.DataFrame(rows)

In [24]:
BASE_DIR = "/mnt/sharedstorage/sabdelazim/"
S5_OUT = os.path.join(BASE_DIR, "S5")
S6_OUT = os.path.join(BASE_DIR, "S6")
OG_folder = os.path.join(BASE_DIR, "images", "all_species_images")

df = pd.concat([
    parse_folder(S5_OUT),
    parse_folder(S6_OUT),
    parse_folder(OG_folder)
], ignore_index=True)

class_per_site = (
    df.groupby(["site", "species"])
      .size()
      .reset_index(name="n_images")
      .sort_values(["site", "n_images"], ascending=[True, False])
)

class_per_site.to_csv(
    "class_per_site.csv",
    index=False
)